In [ ]:
pip install trafilatura spacy pandas httpx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 15.6 MB/s eta 0:00:00


In [ ]:
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 24.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import trafilatura
import json

# Vos URL
urls = [
    "https://en.wikipedia.org/wiki/Flags_of_Oceania",
    "https://en.wikipedia.org/wiki/Flags_of_South_America",
    "https://en.wikipedia.org/wiki/Flags_of_North_America",
    "https://en.wikipedia.org/wiki/Flags_of_Asia",
    "https://en.wikipedia.org/wiki/Flags_of_Africa",
    "https://en.wikipedia.org/wiki/Flags_of_Europe",
    "https://en.wikipedia.org/wiki/National_flag",
    "https://en.wikipedia.org/wiki/Flag",
    "https://en.wikipedia.org/wiki/Vexillology",
    # ... vos autres URL ...
]

data_list = []

print(f"Début du crawling de {len(urls)} pages...")

for url in urls:
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)

        if text:
            # CORRECTION ICI : On compte les mots, pas les caractères
            word_count = len(text.split())

            # On garde seulement si plus de 500 mots
            if word_count > 500:
                entry = {
                    "url": url,
                    "text": text
                }
                data_list.append(entry)
                print(f"✅ Gardé : {url} ({word_count} mots)")
            else:
                print(f"⚠️ Ignoré (trop court) : {url} ({word_count} mots seulement)")
        else:
             print(f"⚠️ Ignoré (pas de texte extrait) : {url}")
    else:
        print(f"❌ Erreur de téléchargement : {url}")

# Sauvegarde
output_file = "crawler_output.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for entry in data_list:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

print(f"\nTerminé ! Données sauvegardées dans '{output_file}'.")

Début du crawling de 9 pages...
✅ Gardé : https://en.wikipedia.org/wiki/Flags_of_Oceania (3541 mots)
✅ Gardé : https://en.wikipedia.org/wiki/Flags_of_South_America (877 mots)
✅ Gardé : https://en.wikipedia.org/wiki/Flags_of_North_America (2201 mots)
✅ Gardé : https://en.wikipedia.org/wiki/Flags_of_Asia (6417 mots)
✅ Gardé : https://en.wikipedia.org/wiki/Flags_of_Africa (9236 mots)
✅ Gardé : https://en.wikipedia.org/wiki/Flags_of_Europe (17140 mots)
✅ Gardé : https://en.wikipedia.org/wiki/National_flag (4413 mots)
✅ Gardé : https://en.wikipedia.org/wiki/Flag (6759 mots)
✅ Gardé : https://en.wikipedia.org/wiki/Vexillology (625 mots)

Terminé ! Données sauvegardées dans 'crawler_output.jsonl'.


In [ ]:
import spacy
import json
import pandas as pd

# 1. CHARGEMENT DU MODÈLE
nlp = spacy.load("en_core_web_trf")

# 2. CHARGEMENT DES DONNÉES
input_file = "crawler_output.jsonl"
data = []
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Analyse de {len(data)} documents...")

extracted_entities = []

# 3. EXTRACTION (NER)
for entry in data:
    url = entry['url']
    text = entry['text']

    # Traitement NLP (peut prendre un peu de temps sur les longs textes)
    doc = nlp(text)

    for ent in doc.ents:
        # On garde les entités pertinentes pour les drapeaux
        # GPE = Lieux/Pays, DATE = Dates, ORG = Organisations, PER = Personnes
        if ent.label_ in ["GPE", "DATE", "ORG", "PERSON", "LOC"]:
            extracted_entities.append({
                "Entity": ent.text,
                "Label": ent.label_,
                "Source_URL": url
            })

# 4. SAUVEGARDE EN CSV (Livrable)
df = pd.DataFrame(extracted_entities)
# Filtrer un peu pour éviter le bruit (optionnel)
df = df.drop_duplicates()

output_csv = "extracted_knowledge.csv"
df.to_csv(output_csv, index=False)

print(f"Terminé ! {len(df)} entités extraites.")
print(f"Fichier sauvegardé : {output_csv}")
# Afficher un aperçu
print(df.head())

Analyse de 9 documents...
Terminé ! 2733 entités extraites.
Fichier sauvegardé : extracted_knowledge.csv
               Entity Label                                      Source_URL
0             Oceania   GPE  https://en.wikipedia.org/wiki/Flags_of_Oceania
1                1908  DATE  https://en.wikipedia.org/wiki/Flags_of_Oceania
2  the Southern Cross   ORG  https://en.wikipedia.org/wiki/Flags_of_Oceania
3                1970  DATE  https://en.wikipedia.org/wiki/Flags_of_Oceania
4                Fiji   GPE  https://en.wikipedia.org/wiki/Flags_of_Oceania


In [ ]:
# Partie 2.2 : Extraction de Relations basique
# On cherche des structures : Entité A (Sujet) -> VERBE -> Entité B (Objet)

relations = []

for entry in data: # 'data' vient de votre étape précédente
    doc = nlp(entry['text'])

    for token in doc:
        # On cherche un verbe
        if token.pos_ == "VERB":
            subj = ""
            obj = ""

            # On regarde les enfants du verbe (sujet et objet)
            for child in token.children:
                if child.dep_ == "nsubj": # Sujet nominal
                    subj = child.text
                if child.dep_ in ["dobj", "attr", "acomp"]: # Objet direct ou attribut
                    obj = child.text

            # Si on a un triplet complet (Sujet + Verbe + Objet)
            if subj and obj:
                relations.append({
                    "Sujet": subj,
                    "Verbe": token.lemma_, # Forme infinitive du verbe
                    "Objet": obj,
                    "Source": entry['url']
                })

# Sauvegarde des relations
df_rel = pd.DataFrame(relations)
print(f"Relations trouvées : {len(df_rel)}")
print(df_rel.head(10))

# Sauvegarder dans un nouveau CSV ou ajouter à l'existant
df_rel.to_csv("extracted_relations.csv", index=False)

Relations trouvées : 492
      Sujet      Verbe         Objet  \
0      that  represent       islands   
1     which   comprise        Tuvalu   
2   chevron       fill         space   
3  triangle     divide         field   
4   article       need     citations   
5      band     recall    federation   
6      band  represent  independence   
7   article       need     citations   
8     White  represent          snow   
9       Red  represent     sacrifice   

                                              Source  
0     https://en.wikipedia.org/wiki/Flags_of_Oceania  
1     https://en.wikipedia.org/wiki/Flags_of_Oceania  
2     https://en.wikipedia.org/wiki/Flags_of_Oceania  
3     https://en.wikipedia.org/wiki/Flags_of_Oceania  
4  https://en.wikipedia.org/wiki/Flags_of_South_A...  
5  https://en.wikipedia.org/wiki/Flags_of_South_A...  
6  https://en.wikipedia.org/wiki/Flags_of_South_A...  
7  https://en.wikipedia.org/wiki/Flags_of_North_A...  
8  https://en.wikipedia.org/wiki/Flags_

In [ ]:
# Assurez-vous d'avoir chargé le modèle
nlp = spacy.load("en_core_web_trf") # ou "fr_core_news_lg"

extracted_entities = []

# Liste noire manuelle pour les mots que vous voyez encore passer
blacklist = ["that", "which", "each", "these", "those", "flag", "flags"]

for entry in data:
    url = entry['url']
    text = entry['text']

    doc = nlp(text)

    for ent in doc.ents:
        # 1. FILTRE PAR TYPE (Consigne: PERSON, ORG, GPE, DATE)
        # On ne garde que les types demandés par le Lab
        if ent.label_ not in ["GPE", "DATE", "ORG", "PERSON"]:
            continue

        # 2. FILTRE STOP WORDS (Consigne: Filter out common nouns)
        # On nettoie le texte de l'entité (minuscule et sans espaces)
        clean_text = ent.text.strip().lower()

        # Si le mot est un "stop word" (le, la, that, which...) -> on ignore
        if nlp.vocab[clean_text].is_stop:
            continue

        # 3. FILTRE TAILLE ET LISTE NOIRE
        # On ignore les mots trop courts (1 ou 2 lettres) ou dans la liste noire
        if len(clean_text) < 3 or clean_text in blacklist:
            continue

        extracted_entities.append({
            "Entity": ent.text,   # Le texte original (avec majuscule)
            "Label": ent.label_,  # Le type (GPE, DATE...)
            "Source_URL": url
        })

# Sauvegarde et vérification
df = pd.DataFrame(extracted_entities)
# On supprime les doublons exacts
df = df.drop_duplicates()

print(f"Après nettoyage, {len(df)} entités valides trouvées.")
print(df.head(10))

# Sauvegarde pour Excel (avec point-virgule)
df.to_csv("extracted_knowledge.csv", sep=';', encoding='utf-8-sig', index=False)

Après nettoyage, 2640 entités valides trouvées.
                                Entity Label  \
0                              Oceania   GPE   
1                                 1908  DATE   
2                   the Southern Cross   ORG   
3                                 1970  DATE   
4                                 Fiji   GPE   
5                                 1979  DATE   
7               the Marshall Islands[4   GPE   
8                                 1978  DATE   
9   the Federated States of Micronesia   GPE   
10                                1968  DATE   

                                        Source_URL  
0   https://en.wikipedia.org/wiki/Flags_of_Oceania  
1   https://en.wikipedia.org/wiki/Flags_of_Oceania  
2   https://en.wikipedia.org/wiki/Flags_of_Oceania  
3   https://en.wikipedia.org/wiki/Flags_of_Oceania  
4   https://en.wikipedia.org/wiki/Flags_of_Oceania  
5   https://en.wikipedia.org/wiki/Flags_of_Oceania  
7   https://en.wikipedia.org/wiki/Flags_of_Oceania  